# Part 1, Question 1: Campaign Team Tracking and Coverage Reconciliation

## Notebook 1: Data Ingestion

### Objective

This notebook builds the reproducible data ingestion pipeline for the campaign tracking assessment. It imports all raw datasets, validates the file structure, combines the daily GPS track files into a single dataset, converts spatial datasets into GeoDataFrames where appropriate, and exports the processed data into a single spatial data store for downstream quality assurance and analysis.

The notebook performs no cleaning or analytical transformations. Its sole purpose is to create a reproducible and reusable data ingestion workflow.

## IMPORT REQUIRED LIBRARIES

### This notebook builds the reproducible data ingestion pipeline for the campaign tracking assessment. It imports all raw datasets, validates the file structure, combines the daily GPS track files into a single dataset, converts spatial datasets into GeoDataFrames where appropriate, and exports the processed data into a single spatial data store for downstream quality assurance and analysis.

The notebook performs no cleaning or analytical transformations. Its sole purpose is to create a reproducible and reusable data ingestion workflow.

In [1]:
import sys
from pathlib import Path
import fiona

import pandas as pd
import geopandas as gpd
import shapely

In [2]:
# =============================================================================
# SOFTWARE ENVIRONMENT
# =============================================================================
# Recording package versions supports reproducibility by documenting the
# software environment used for the analysis.

print(f"Python:    {sys.version.split()[0]}")
print(f"Pandas:    {pd.__version__}")
print(f"GeoPandas: {gpd.__version__}")
print(f"Shapely:   {shapely.__version__}")

Python:    3.11.10
Pandas:    2.2.3
GeoPandas: 1.0.1
Shapely:   2.1.1


In [3]:
#Beore inporting my datasets i started with the readme which servea like the metadata , give me the exact explanation to the dataset
# =============================================================================
###  DATA DESCRIPTION
# =============================================================================
# Before importing the datasets, the project documentation (README.md) was
# reviewed. The README serves as the metadata for this assessment by describing
# the purpose of the project, the available datasets, their structure, and the
# expected analytical tasks. Reviewing the metadata first provides the
# necessary context for understanding each dataset before analysis begins.


# =============================================================================
# DEFINE PROJECT DIRECTORY
# =============================================================================
# Relative paths are used instead of absolute file paths or GitHub URLs.
# This is considered best practice for reproducible research because anyone
# who clones the repository can run the notebook without changing file paths.

PROJECT_ROOT = Path.cwd()

# Path to the main data directory
DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "eHA_Assessment_Data_Pack_v4_CANDIDATE"
    / "Part1_Q1_Campaign_Tracking"
)

# Display the resolved data directory to verify the path is correct.
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

Project root: C:\Users\Idris\Downloads\Senior_Coordinator_Data_and_GIS_Analytics
Data directory: C:\Users\Idris\Downloads\Senior_Coordinator_Data_and_GIS_Analytics\data\raw\eHA_Assessment_Data_Pack_v4_CANDIDATE\Part1_Q1_Campaign_Tracking


In [4]:
# =============================================================================
# DEFINE DATASET FILE PATHS
# =============================================================================

BOUNDARIES_FILE = DATA_DIR / "boundaries.gpkg"
ETALLY_FILE = DATA_DIR / "etally_daily.csv"
SETTLEMENTS_FILE = DATA_DIR / "settlement_masterlist.csv"
INACCESSIBLE_FILE = DATA_DIR / "inaccessible_settlements.csv"

# Folder containing all GPS track CSV files
TRACKS_DIR = DATA_DIR / "tracks"

In [5]:
# =============================================================================
# IMPORT MAIN DATASETS
# =============================================================================

etally = pd.read_csv(ETALLY_FILE)
settlements = pd.read_csv(SETTLEMENTS_FILE)
inaccessible = pd.read_csv(INACCESSIBLE_FILE)
boundaries = gpd.read_file(BOUNDARIES_FILE)

C:\Users\Idris\anaconda3\envs\sds2026\Lib\site-packages\pyogrio\geopandas.py:265: UserWarning: More than one layer found in 'boundaries.gpkg': 'wards' (default), 'lgas', 'state'. Specify layer parameter to avoid this warning.
  result = read_func(


In [6]:
# =============================================================================
# SINCE WE HAVE A GEODATABASE LETS IMPORT ALL THE ADMINISTRATIVE BOUNDARY LAYERS
# =============================================================================

wards = gpd.read_file(BOUNDARIES_FILE, layer="wards")
lgas = gpd.read_file(BOUNDARIES_FILE, layer="lgas")
state = gpd.read_file(BOUNDARIES_FILE, layer="state")

In [65]:
# =============================================================================
# IMPORT AND COMBINE ALL GPS TRACK FILES
# =============================================================================
# Each GPS track is stored as a separate CSV file (one file per team per day).
# The files are combined into a single DataFrame to create a unified dataset
# for downstream quality assurance, spatial processing, and analysis.
#
# A new column (track_file) records the source filename, ensuring that every
# observation can be traced back to its original file.

track_files = sorted(TRACKS_DIR.glob("*.csv"))

tracks_raw = pd.concat(
    [
        pd.read_csv(file).assign(track_file=file.stem)
        for file in track_files
    ],
    ignore_index=True,
)

print(f"Loaded {len(track_files)} GPS track files.")
print(f"Combined dataset shape: {tracks_raw.shape}")

Loaded 160 GPS track files.
Combined dataset shape: (956702, 8)


In [66]:
# =============================================================================
# CONVERT TRACKS TO A GEOSPATIAL DATAFRAME
# =============================================================================
# GPS coordinates are converted into point geometries using the WGS84
# geographic coordinate reference system (EPSG:4326).

tracks_gdf = gpd.GeoDataFrame(
    tracks_raw,
    geometry=gpd.points_from_xy(
        tracks_raw["longitude"],
        tracks_raw["latitude"]
    ),
    crs="EPSG:4326"
)

print(tracks_gdf.shape)

(956702, 9)


### =============================================================================
### VERIFY SUCCESSFUL DATA INGESTION
### =============================================================================
### Confirm that each dataset has been successfully imported and report its
### dimensions. This provides a quick validation that the ingestion pipeline
### completed as expected before proceeding to quality assurance and spatial
### analysis.

In [67]:

print(f"Daily campaign records:          {etally.shape}")
print(f"Settlement master list:          {settlements.shape}")
print(f"Inaccessible settlements:        {inaccessible.shape}")
print(f"Ward boundaries:                 {wards.shape}")
print(f"LGA boundaries:                  {lgas.shape}")
print(f"State boundary:                  {state.shape}")
print(f"Combined GPS track dataset:      {tracks_raw.shape}")

Daily campaign records:          (2023, 7)
Settlement master list:          (2562, 10)
Inaccessible settlements:        (75, 7)
Ward boundaries:                 (40, 7)
LGA boundaries:                  (4, 5)
State boundary:                  (1, 2)
Combined GPS track dataset:      (956702, 8)


### =============================================================================
### EXPORT PROCESSED DATA TO A SINGLE SPATIALLY ENABLED STORE --> -->
### =============================================================================

* A GeoPackage was selected because it is an Open Geospatial Consortium (OGC) standard capable of storing multiple spatial layers within a single portable file. It is fully supported by GeoPandas, QGIS and ArcGIS and therefore provides a reproducible and interoperable spatial data store..
###
### To ensure the pipeline is idempotent, the existing GeoPackage is removed
### before being recreated. Re-running this notebook therefore replaces the
### outputs rather than appending duplicate records.
### =============================================================================



In [68]:
# -------------------------------------------------------------------------
# Create output directory
# -------------------------------------------------------------------------

processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

gpkg_path = processed_dir / "campaign_data.gpkg"

In [69]:
# -------------------------------------------------------------------------
# Ensure settlement masterlist is spatially enabled
# -------------------------------------------------------------------------

settlements_gdf = gpd.GeoDataFrame(
    settlements.copy(),
    geometry=gpd.points_from_xy(
        settlements["longitude"],
        settlements["latitude"]
    ),
    crs="EPSG:4326"
)

In [70]:
# -------------------------------------------------------------------------
# Ensure an idempotent export
# -------------------------------------------------------------------------

if gpkg_path.exists():
    gpkg_path.unlink()

## Pipeline Summary

This notebook implements the data ingestion component of the analysis. Raw GPS track files from all campaign teams are combined into a single dataset, converted into a spatially enabled GeoDataFrame, and exported together with the administrative and settlement reference layers into a GeoPackage. The workflow is designed to be reproducible and idempotent so that rerunning the notebook regenerates the spatial data store without creating duplicate records.

In [71]:
# -------------------------------------------------------------------------
# Export spatial layers
# -------------------------------------------------------------------------

tracks_gdf.to_file(
    gpkg_path,
    layer="gps_tracks",
    driver="GPKG"
)

settlements_gdf.to_file(
    gpkg_path,
    layer="settlements",
    driver="GPKG"
)

wards.to_file(
    gpkg_path,
    layer="wards",
    driver="GPKG"
)

lgas.to_file(
    gpkg_path,
    layer="lgas",
    driver="GPKG"
)

state.to_file(
    gpkg_path,
    layer="state",
    driver="GPKG"
)

print("✓ Spatial data store successfully created.")
print(f"GeoPackage location:\n{gpkg_path}")

✓ Spatial data store successfully created.
GeoPackage location:
C:\Users\Idris\Downloads\Senior_Coordinator_Data_and_GIS_Analytics\data\processed\campaign_data.gpkg


In [72]:
# =============================================================================
# VERIFY EXPORTED GEOPACKAGE
# =============================================================================

import fiona

layers = fiona.listlayers(gpkg_path)

print("Layers written to GeoPackage:\n")

for layer in layers:
    print(f"• {layer}")

print(f"\nTotal layers: {len(layers)}")

Layers written to GeoPackage:

• gps_tracks
• settlements
• wards
• lgas
• state

Total layers: 5
